In [1]:
from src.donnees import import_donnees, renomer_cle_jointure, concatenation_annees
from src.nettoyage import recodage, mapping_renommer_colonnes, colonnes_a_supprimer, création_age_usager, jointure

Nous avons réalisé des analyses préliminaires sur nos données (disponible dans le fichier pre-processing), cela nous a permis de relever certains problèmes dans les données que nous avons pris en compte pour le nettoyage de nos données ci-dessous (comme les na, les noms de colonnes, les doublons ...)

In [2]:
donnees_completes = import_donnees()

donnees_completes["caract"][22] = renomer_cle_jointure(donnees_completes["caract"][22], "Num_Acc", "Accident_Id")

# CARACTERISTIQUE
df_caract = concatenation_annees(donnees_completes, "caract")
# LIEUX
df_lieux = concatenation_annees(donnees_completes, "lieux")
# VEHICULE
df_vehicule = concatenation_annees(donnees_completes, "vehicule")
# USAGER
df_usager = concatenation_annees(donnees_completes, "usager")

/home/onyxia/work/Projet_pythonDS/src/donnees.py:5: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(nom_fichier_csv, sep=';', encoding='UTF-8')
/home/onyxia/work/Projet_pythonDS/src/donnees.py:5: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(nom_fichier_csv, sep=';', encoding='UTF-8')


In [3]:
mappings = mapping_renommer_colonnes()

In [4]:
# recodage des noms des colonnes 
df_caract_recoder = recodage(df_caract, mappings["caract"])
df_lieux_recoder = recodage(df_lieux, mappings["lieux"])
df_vehicule_recoder = recodage(df_vehicule, mappings["vehicule"])
df_usager_recoder = recodage(df_usager, mappings["usager"])

# supprimer les doublons de corrections des données dans le fichier lieux 
df_lieux_recoder = df_lieux_recoder.drop_duplicates(subset="Num_Acc", keep="last")

# age pour les usagers
df_usager_recoder = création_age_usager(df_usager_recoder)

In [5]:
df_final = jointure(df_caract_recoder, df_lieux_recoder, df_vehicule_recoder, df_usager_recoder)

In [6]:
colonnes_a_supprimer = colonnes_a_supprimer()
df_final = df_final.drop(columns=colonnes_a_supprimer, errors="ignore")

In [7]:
df_final.columns

Index(['Num_Acc', 'jour', 'mois', 'an', 'hrmn', 'lum', 'dep', 'com', 'agg',
       'int', 'atm', 'col', 'adr', 'lat', 'long', 'catr', 'voie', 'v1', 'v2',
       'circ', 'nbv', 'vosp', 'prof', 'pr', 'pr1', 'plan', 'lartpc', 'larrout',
       'surf', 'infra', 'situ', 'vma', 'id_vehicule', 'catv', 'obs', 'obsm',
       'choc', 'manv', 'id_usager', 'catu', 'grav', 'sexe', 'trajet', 'secu1',
       'secu2', 'secu3', 'age'],
      dtype='object')

In [8]:
df_final

,Num_Acc,jour,mois,an,hrmn,lum,dep,com,agg,int,...,manv,id_usager,catu,grav,sexe,trajet,secu1,secu2,secu3,age
0,202400000001,25,mars,2024,07:40,Crépuscule ou aube,70,70285,Hors agglomération,Hors intersection,...,Déporté,203 988 581,Conducteur,Blessé hospitalisé,Homme,Domicile - École,Ceinture,Non renseigné,Non renseigné,23
1,202400000001,25,mars,2024,07:40,Crépuscule ou aube,70,70285,Hors agglomération,Hors intersection,...,Manœuvre d’évitement,203 988 582,Conducteur,Indemne,Homme,Utilisation professionnelle,Ceinture,Non renseigné,Non renseigné,29
2,202400000002,20,mars,2024,15:05,Plein jour,21,21054,En agglomération,Intersection en T,...,Tournant,203 988 579,Piéton,Blessé hospitalisé,Femme,Promenade - loisirs,Aucun équipement,Non renseigné,Non renseigné,99
3,202400000002,20,mars,2024,15:05,Plein jour,21,21054,En agglomération,Intersection en T,...,Tournant,203 988 580,Conducteur,Indemne,Homme,Utilisation professionnelle,Ceinture,Aucun équipement,Non renseigné,39
4,202400000003,22,mars,2024,19:30,Crépuscule ou aube,15,15012,Hors agglomération,Hors intersection,...,Sans changement de direction,203 988 574,Passager,Blessé léger,Femme,Promenade - loisirs,Non déterminable,Aucun équipement,Non renseigné,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
377697,202200055301,1,janvier,2022,08:40,Plein jour,81,81099,Hors agglomération,Intersection en T,...,Traversant la chaussée,968 230,Conducteur,Indemne,Femme,Promenade - loisirs,Ceinture,Non renseigné,Non renseigné,24
377698,202200055301,1,janvier,2022,08:40,Plein jour,81,81099,Hors agglomération,Intersection en T,...,Traversant la chaussée,968 231,Passager,Blessé hospitalisé,Femme,Promenade - loisirs,Ceinture,Non renseigné,Non renseigné,22
377699,202200055301,1,janvier,2022,08:40,Plein jour,81,81099,Hors agglomération,Intersection en T,...,Sans changement de direction,968 232,Conducteur,Blessé léger,Femme,Promenade - loisirs,Ceinture,Non renseigné,Non renseigné,73
377700,202200055302,1,mars,2022,16:55,Plein jour,41,41018,En agglomération,Hors intersection,...,Sans changement de direction,968 228,Conducteur,Blessé hospitalisé,Homme,Domicile - Travail,Casque,Gants (2RM/3RM),Non renseigné,34


In [9]:
df_final["id_usager"].nunique()
# le nombre de usager devrait etre le meme nombre que le nb de lignes dans df_final ?

377638

In [10]:
df_final[df_final.duplicated(["Num_Acc", "id_vehicule", "id_usager"], keep=False)]
# pas de doublons ! 

,Num_Acc,jour,mois,an,hrmn,lum,dep,com,agg,int,...,manv,id_usager,catu,grav,sexe,trajet,secu1,secu2,secu3,age


In [11]:
df_final[df_final.duplicated("id_usager", keep=False)]


,Num_Acc,jour,mois,an,hrmn,lum,dep,com,agg,int,...,manv,id_usager,catu,grav,sexe,trajet,secu1,secu2,secu3,age
10,202400000004,24,mars,2024,17:50,Plein jour,14,14118,En agglomération,Intersection en T,...,Tournant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
1111,202400000466,30,avril,2024,19:20,Plein jour,79,79191,En agglomération,Intersection à plus de 4 branches,...,Sans changement de direction,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
6665,202400002915,20,décembre,2024,16:00,Plein jour,62,62758,En agglomération,Intersection en T,...,Arrêté (hors stationnement),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
11558,202400005069,5,juin,2024,11:44,Plein jour,2B,2B033,En agglomération,Hors intersection,...,Arrêté (hors stationnement),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
15975,202400007018,3,juin,2024,14:50,Plein jour,49,49007,En agglomération,Hors intersection,...,Inconnue,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
227810,202300044719,1,octobre,2023,16:35,Plein jour,60,60057,En agglomération,Hors intersection,...,A contresens,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
229837,202300045599,11,mai,2023,10:02,Plein jour,75,75117,En agglomération,Intersection en X,...,Sans changement de direction,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
240635,202300050308,9,mai,2023,20:40,Plein jour,14,14333,En agglomération,Hors intersection,...,Sans changement de direction,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
242169,202300050996,4,octobre,2023,06:27,Nuit sans éclairage public,68,68286,Hors agglomération,Intersection en T,...,Tournant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>


In [12]:
df_final[df_final["Num_Acc"]==202400000004]

,Num_Acc,jour,mois,an,hrmn,lum,dep,com,agg,int,...,manv,id_usager,catu,grav,sexe,trajet,secu1,secu2,secu3,age
9,202400000004,24,mars,2024,17:50,Plein jour,14,14118,En agglomération,Intersection en T,...,Inconnue,203 988 573,Conducteur,Blessé léger,Homme,Promenade - loisirs,Casque,Gants (2RM/3RM),Non renseigné,63
10,202400000004,24,mars,2024,17:50,Plein jour,14,14118,En agglomération,Intersection en T,...,Tournant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>


In [13]:
df_usager[df_usager["Num_Acc"]==202400000004]

,Num_Acc,id_usager,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
9,202400000004,203 988 573,155 781 754,B01,1,1,4,1,1963.0,5,2,6,-1,0,0,-1


In [14]:
df_vehicule[df_vehicule["Num_Acc"]==202400000004]

,Num_Acc,id_vehicule,num_veh,senc,catv,obs,obsm,choc,manv,motor,occutc
4,202400000004,155 781 754,B01,3,34,0,2,0,0,1,NaN
5,202400000004,155 781 755,A01,3,7,0,2,3,15,0,NaN


In [15]:
df_caract_recoder

,Num_Acc,jour,mois,an,hrmn,lum,dep,com,agg,int,atm,col,adr,lat,long
0,202400000001,25,mars,2024,07:40,Crépuscule ou aube,70,70285,Hors agglomération,Hors intersection,Brouillard - fumée,Deux véhicules - frontale,D438,"47,56277000","6,75832000"
1,202400000002,20,mars,2024,15:05,Plein jour,21,21054,En agglomération,Intersection en T,Temps éblouissant,Autre collision,HOTEL DIEU (RUE DE L'),"47,02109000","4,83755000"
2,202400000003,22,mars,2024,19:30,Crépuscule ou aube,15,15012,Hors agglomération,Hors intersection,Normale,Autre collision,Allée des Tilleuls,"44,90238400","2,49641800"
3,202400000004,24,mars,2024,17:50,Plein jour,14,14118,En agglomération,Intersection en T,Temps éblouissant,Deux véhicules - par le côté,128 Rue d'Authie,"49,19166000","-0,39851000"
4,202400000005,25,mars,2024,19:35,Nuit avec éclairage public allumé,13,13106,Hors agglomération,Intersection en T,Pluie légère,Trois véhicules - collisions multiples,BEDOULE (CHEMIN DE LA),"43,39000000","5,35000000"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164521,202200055298,1,janvier,2022,03:50,Nuit sans éclairage public,2B,2B293,Hors agglomération,Hors intersection,Normale,Autre collision,D71,"42,3101650000","9,4785830000"
164522,202200055299,1,janvier,2022,07:20,Nuit sans éclairage public,84,84074,Hors agglomération,Hors intersection,Normale,Autre collision,D973,"43,7531640000","5,2244760000"
164523,202200055300,1,janvier,2022,04:27,Nuit sans éclairage public,74,74001,Hors agglomération,Hors intersection,Autre,Autre collision,D22,"46,2825320000","6,7328060000"
164524,202200055301,1,janvier,2022,08:40,Plein jour,81,81099,Hors agglomération,Intersection en T,Normale,Deux véhicules - par le côté,Chemin Toulze,"43,9272650000","1,9156370000"


PB : les usagers de certains vehicule ne sont pas repertorié dans les données, comme nous nous interressons aux usgaers car nous regardons notamment les blessures des usagers ne pas avoir d'informations sur eux = pas possible de faire les analyses. Ces usagers représente 64 sur 377638 usagers, nous pouvons donc les supprimés sans nuire à nos analyses, car ils sont peu nombreux.

In [16]:
df_lieux_recoder

,Num_Acc,catr,voie,v1,v2,circ,nbv,vosp,prof,pr,pr1,plan,lartpc,larrout,surf,infra,situ,vma
0,202400000001,Route départementale,D438,0,NaN,Bidirectionnelle,2,Sans objet,Plat,1,260,En courbe à gauche,NaN,7,Normale,Aucun,Sur chaussée,90
2,202400000002,Voie communale,POTERNE (RUE),0,NaN,À sens unique,1,Sans objet,Plat,-1,-1,Partie rectiligne,NaN,-1,Autre,Aucun,Sur chaussée,30
3,202400000003,Voie communale,TILLEULS (ALLEE DES),0,NaN,Bidirectionnelle,2,Sans objet,Plat,-1,-1,Partie rectiligne,NaN,-1,Normale,Aucun,Sur accotement,50
5,202400000004,Voie communale,HAGUE (RUE DE LA),0,NaN,Bidirectionnelle,2,Sans objet,Plat,-1,-1,Partie rectiligne,NaN,-1,Normale,Aucun,Sur chaussée,50
7,202400000005,Autoroute,BEDOULE (CHEMIN DE LA),0,NaN,Bidirectionnelle,4,Piste cyclable,Pente,272,0,Partie rectiligne,NaN,17,Mouillée,Aucun,Sur chaussée,70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
196405,202200055298,Route départementale,71,-1,NaN,Bidirectionnelle,2,Sans objet,Pente,(1),(1),Partie rectiligne,NaN,-1,Normale,Aucun,Autre,80
196406,202200055299,Route départementale,973,-1,NaN,Bidirectionnelle,2,Sans objet,Plat,29,0,En courbe à gauche,NaN,-1,Normale,Aucun,Sur accotement,80
196407,202200055300,Route départementale,22,0,D,Bidirectionnelle,2,Sans objet,Plat,39,553,En courbe à gauche,NaN,-1,Verglacée,Aucun,Sur accotement,80
196408,202200055301,Route départementale,18,-1,D,Bidirectionnelle,2,Sans objet,Plat,30,125,Partie rectiligne,NaN,-1,Normale,Aucun,Sur chaussée,80


In [19]:
df_vehicule_recoder

,Num_Acc,id_vehicule,num_veh,senc,catv,obs,obsm,choc,manv,motor,occutc
0,202400000001,155 781 758,A01,1,VL,Sans objet,Véhicule,Avant,Déporté,1,NaN
1,202400000001,155 781 759,B01,2,PL,Sans objet,Véhicule,Avant droit,Manœuvre d’évitement,1,NaN
2,202400000002,155 781 757,A01,1,VU,Sans objet,Piéton,Avant gauche,Tournant,1,NaN
3,202400000003,155 781 756,A01,3,Voiturette,Mobilier urbain,Piéton,Avant,Sans changement de direction,1,NaN
4,202400000004,155 781 754,B01,3,Scooter,Sans objet,Véhicule,Aucun,Inconnue,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...
280751,202200055300,715 633,A01,2,VL,Arbre,Aucun,Avant,Sans changement de direction,1,NaN
280752,202200055301,715 631,A01,2,VL,Sans objet,Aucun,Côté gauche,Traversant la chaussée,1,NaN
280753,202200055301,715 632,B01,2,VL,Sans objet,Véhicule,Avant,Sans changement de direction,1,NaN
280754,202200055302,715 629,A01,1,Motocyclette,Sans objet,Véhicule,Avant,Sans changement de direction,1,NaN


In [21]:
df_usager_recoder

,Num_Acc,id_usager,id_vehicule,num_veh,place,catu,grav,sexe,trajet,secu1,secu2,secu3,locp,actp,etatp,age
0,202400000001,203 988 581,155 781 758,A01,1,Conducteur,Blessé hospitalisé,Homme,Domicile - École,Ceinture,Non renseigné,Non renseigné,-1,-1,-1,23
1,202400000001,203 988 582,155 781 759,B01,1,Conducteur,Indemne,Homme,Utilisation professionnelle,Ceinture,Non renseigné,Non renseigné,-1,-1,-1,29
2,202400000002,203 988 579,155 781 757,A01,10,Piéton,Blessé hospitalisé,Femme,Promenade - loisirs,Aucun équipement,Non renseigné,Non renseigné,3,3,1,99
3,202400000002,203 988 580,155 781 757,A01,1,Conducteur,Indemne,Homme,Utilisation professionnelle,Ceinture,Aucun équipement,Non renseigné,3,3,1,39
4,202400000003,203 988 574,155 781 756,A01,2,Passager,Blessé léger,Femme,Promenade - loisirs,Non déterminable,Aucun équipement,Non renseigné,-1,-1,-1,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
377633,202200055301,968 230,715 631,A01,1,Conducteur,Indemne,Femme,Promenade - loisirs,Ceinture,Non renseigné,Non renseigné,0,0,-1,24
377634,202200055301,968 231,715 631,A01,8,Passager,Blessé hospitalisé,Femme,Promenade - loisirs,Ceinture,Non renseigné,Non renseigné,0,0,-1,22
377635,202200055301,968 232,715 632,B01,1,Conducteur,Blessé léger,Femme,Promenade - loisirs,Ceinture,Non renseigné,Non renseigné,0,0,-1,73
377636,202200055302,968 228,715 629,A01,1,Conducteur,Blessé hospitalisé,Homme,Domicile - Travail,Casque,Gants (2RM/3RM),Non renseigné,-1,-1,-1,34


In [17]:
# verifier si pas de création de na bizarre quelque part 

In [20]:
df_lieux_recoder["Num_Acc"].nunique()

164526

In [23]:
df_vehicule_recoder[["Num_Acc", "id_vehicule"]].nunique()

Num_Acc        164526
id_vehicule    280756
dtype: int64

In [ ]:
df_vehicule_recoder[df_vehicule_recoder.duplicated(["Num_Acc", "id_vehicule"], keep=False)] # pas de doublon

,Num_Acc,id_vehicule,num_veh,senc,catv,obs,obsm,choc,manv,motor,occutc


In [ ]:
df_usager_recoder[["Num_Acc", "id_vehicule", "id_usager"]].nunique() # moins de vehicule que dans vehoicule bien le fair des usager pas repertorié 

Num_Acc        164526
id_vehicule    280692
id_usager      377638
dtype: int64

In [ ]:
print(df_final["Num_Acc"].isna().sum())
print(df_final["id_vehicule"].isna().sum())
print(df_final["id_usager"].isna().sum())   # nan pour les usager seulement à supprimer !!!  nombre d'usager = 377638 = nb de ligne dans df_final (a voir d'autre colonne suppression de na ou pas ?)


0
0
64


In [33]:
df_final = df_final.dropna(subset=["id_usager"])

In [34]:
df_final

,Num_Acc,jour,mois,an,hrmn,lum,dep,com,agg,int,...,manv,id_usager,catu,grav,sexe,trajet,secu1,secu2,secu3,age
0,202400000001,25,mars,2024,07:40,Crépuscule ou aube,70,70285,Hors agglomération,Hors intersection,...,Déporté,203 988 581,Conducteur,Blessé hospitalisé,Homme,Domicile - École,Ceinture,Non renseigné,Non renseigné,23
1,202400000001,25,mars,2024,07:40,Crépuscule ou aube,70,70285,Hors agglomération,Hors intersection,...,Manœuvre d’évitement,203 988 582,Conducteur,Indemne,Homme,Utilisation professionnelle,Ceinture,Non renseigné,Non renseigné,29
2,202400000002,20,mars,2024,15:05,Plein jour,21,21054,En agglomération,Intersection en T,...,Tournant,203 988 579,Piéton,Blessé hospitalisé,Femme,Promenade - loisirs,Aucun équipement,Non renseigné,Non renseigné,99
3,202400000002,20,mars,2024,15:05,Plein jour,21,21054,En agglomération,Intersection en T,...,Tournant,203 988 580,Conducteur,Indemne,Homme,Utilisation professionnelle,Ceinture,Aucun équipement,Non renseigné,39
4,202400000003,22,mars,2024,19:30,Crépuscule ou aube,15,15012,Hors agglomération,Hors intersection,...,Sans changement de direction,203 988 574,Passager,Blessé léger,Femme,Promenade - loisirs,Non déterminable,Aucun équipement,Non renseigné,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
377697,202200055301,1,janvier,2022,08:40,Plein jour,81,81099,Hors agglomération,Intersection en T,...,Traversant la chaussée,968 230,Conducteur,Indemne,Femme,Promenade - loisirs,Ceinture,Non renseigné,Non renseigné,24
377698,202200055301,1,janvier,2022,08:40,Plein jour,81,81099,Hors agglomération,Intersection en T,...,Traversant la chaussée,968 231,Passager,Blessé hospitalisé,Femme,Promenade - loisirs,Ceinture,Non renseigné,Non renseigné,22
377699,202200055301,1,janvier,2022,08:40,Plein jour,81,81099,Hors agglomération,Intersection en T,...,Sans changement de direction,968 232,Conducteur,Blessé léger,Femme,Promenade - loisirs,Ceinture,Non renseigné,Non renseigné,73
377700,202200055302,1,mars,2022,16:55,Plein jour,41,41018,En agglomération,Hors intersection,...,Sans changement de direction,968 228,Conducteur,Blessé hospitalisé,Homme,Domicile - Travail,Casque,Gants (2RM/3RM),Non renseigné,34
